# Data Extraction After Fine-tuning

Colab notebook for extraction experiments on fine-tuned `Florence-2` and `Qwen2-VL` checkpoints.

## 0. Setup

In [ ]:
%pip install -q huggingface_hub comet_ml python-dotenv matplotlib peft bitsandbytes qwen-vl-utils

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

from huggingface_hub import snapshot_download


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return None


def get_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name)


project_root = find_project_root(Path.cwd())
repo_url = get_colab_secret('COURSE_WORK2026_REPO_URL')

if project_root is None:
    clone_dir = Path('/content/course_work2026')
    if not (clone_dir / '.git').exists():
        if not repo_url:
            raise RuntimeError('Could not detect the repository locally and COURSE_WORK2026_REPO_URL is not set in Colab secrets.')
        subprocess.run(['git', 'clone', repo_url, str(clone_dir)], check=True)
    project_root = clone_dir

sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

from src.colab_setup import setup_colab

setup_summary, bootstrap_experiment = setup_colab(repo_url=repo_url)
bootstrap_experiment.set_name('extraction-bootstrap')
bootstrap_experiment.end()

PROJECT_ROOT = Path(setup_summary['project_root'])
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
EXTRACTION_OUTPUT_DIR = ARTIFACTS_ROOT / 'privacy_attacks' / 'extraction'
CHECKPOINTS_ROOT = PROJECT_ROOT / 'checkpoints'
HF_CHECKPOINT_REPO = os.getenv('HF_MODEL_REPO') or 'karimkramin/docvqa-privacy-checkpoints'

EXTRACTION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_ROOT.mkdir(parents=True, exist_ok=True)

snapshot_path = snapshot_download(repo_id=HF_CHECKPOINT_REPO, repo_type='model', local_dir=str(CHECKPOINTS_ROOT), token=os.getenv('HF_TOKEN'))

print(json.dumps(setup_summary, ensure_ascii=False, indent=2))
print({'project_root': str(PROJECT_ROOT), 'checkpoints_root': str(CHECKPOINTS_ROOT), 'snapshot_path': str(snapshot_path), 'extraction_output_dir': str(EXTRACTION_OUTPUT_DIR)})

## 1. Extraction Utilities

In [ ]:
from __future__ import annotations

import gc
import re
from pathlib import Path

import comet_ml
import matplotlib.pyplot as plt
import pandas as pd
import torch
from peft import PeftModel
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration

from src.docqa_metrics import best_metric_over_answers, build_answer_pool, estimate_random_baseline
from src.inference_scenarios import generate_scenario_image, generate_scenario_ocr


device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

IMAGE_ONLY_SCENARIOS = ['original', 'img_black', 'img_white', 'img_blur_10', 'img_blur_20', 'img_blur_50']
QWEN_IMAGE_OCR_SCENARIOS = [
    'img_black+ocr_drop_k0',
    'img_black+ocr_drop_k20',
    'img_black+ocr_mask_k0',
    'img_black+ocr_mask_k20',
    'img_white+ocr_drop_k0',
    'img_white+ocr_mask_k0',
    'img_blur_20+ocr_drop_k0',
    'img_blur_20+ocr_mask_k0',
    'img_none+ocr_drop_k0',
    'img_none+ocr_mask_k0',
    'img_black+ocr_none',
]
BASELINE_SPECS = [
    {'tag': 'florence2', 'display_name': 'Florence-2-base', 'kind': 'florence', 'base_model_name': 'microsoft/Florence-2-base'},
    {'tag': 'qwen2b', 'display_name': 'Qwen2-VL-2B-Instruct', 'kind': 'qwen2vl', 'base_model_name': 'Qwen/Qwen2-VL-2B-Instruct'},
    {'tag': 'qwen7b', 'display_name': 'Qwen2-VL-7B-Instruct', 'kind': 'qwen2vl', 'base_model_name': 'Qwen/Qwen2-VL-7B-Instruct'},
]
FINETUNED_SPECS = [
    {'tag': 'florence2', 'display_name': 'Florence-2-base', 'kind': 'florence', 'base_model_name': 'microsoft/Florence-2-base', 'final_checkpoint': CHECKPOINTS_ROOT / 'florence2' / 'epoch_50', 'epochs': [1, 3, 10, 30, 50]},
    {'tag': 'qwen2b', 'display_name': 'Qwen2-VL-2B-Instruct', 'kind': 'qwen2vl', 'base_model_name': 'Qwen/Qwen2-VL-2B-Instruct', 'final_checkpoint': CHECKPOINTS_ROOT / 'qwen2b' / 'epoch_30', 'epochs': [1, 3, 10, 30]},
    {'tag': 'qwen7b', 'display_name': 'Qwen2-VL-7B-Instruct', 'kind': 'qwen2vl', 'base_model_name': 'Qwen/Qwen2-VL-7B-Instruct', 'final_checkpoint': CHECKPOINTS_ROOT / 'qwen7b' / 'epoch_30', 'epochs': [1, 3, 10, 30]},
]


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as fh:
        for line in fh:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def load_records(manifest_path: Path, split: str, limit: int | None = None) -> list[dict]:
    records = load_jsonl(manifest_path)
    if limit is not None:
        records = records[:limit]
    image_dir = manifest_path.parent / 'images' / 'original'
    normalized = []
    for idx, record in enumerate(records):
        image_name = Path(record['image_path']).name
        updated = dict(record)
        updated['image_path'] = str(image_dir / image_name)
        updated['split'] = split
        updated['record_key'] = f"{split}:{record.get('example_id', 'na')}:{record.get('local_row_id', idx)}"
        normalized.append(updated)
    return normalized


TRAIN_RECORDS = load_records(ARTIFACTS_ROOT / 'docqa_recovery' / 'benchmark_train' / 'manifest.jsonl', split='seen', limit=800)
VALIDATION_RECORDS = load_records(ARTIFACTS_ROOT / 'docqa_recovery' / 'benchmark' / 'manifest.jsonl', split='unseen', limit=800)
ALL_RECORDS = TRAIN_RECORDS + VALIDATION_RECORDS
RECORD_LOOKUP = {record['record_key']: record for record in ALL_RECORDS}
ANSWER_POOLS = {
    'seen': build_answer_pool(TRAIN_RECORDS),
    'unseen': build_answer_pool(VALIDATION_RECORDS),
}


def mark_model_kind(model, kind: str):
    setattr(model, '_docvqa_model_kind', kind)
    return model


def get_model_kind(model) -> str:
    return getattr(model, '_docvqa_model_kind', 'unknown')


def sanitize_prediction_text(text: str) -> str:
    text = str(text or '')
    text = re.sub(r'<[^>]+>', ' ', text)
    text = ' '.join(text.replace('\n', ' ').split()).strip()
    return text


def load_florence(checkpoint_path: Path | None = None):
    model_name = str(checkpoint_path) if checkpoint_path is not None else 'microsoft/Florence-2-base'
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch_dtype).to(device)
    model.eval()
    return mark_model_kind(model, 'florence'), processor


def load_qwen(base_model_name: str, adapter_path: Path | None = None):
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    processor = AutoProcessor.from_pretrained(base_model_name, trust_remote_code=True)
    model = Qwen2VLForConditionalGeneration.from_pretrained(base_model_name, trust_remote_code=True, torch_dtype=torch.float16, quantization_config=quantization_config, device_map='auto')
    if adapter_path is not None:
        model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()
    return mark_model_kind(model, 'qwen2vl'), processor


def build_qwen_prompt(question: str, mode: str, redacted_ocr: str | None = None) -> str:
    if mode == 'image_only':
        return f"Look at this document image. Answer the question. Give only the answer, nothing else.\nQuestion: {question}"
    return (
        'Here is a document image and its OCR text. Some information has been redacted. '
        'Answer the question with only the answer value, nothing else.\n\n'
        f'OCR text: {redacted_ocr or ""}\n\nQuestion: {question}'
    )


def generate_prediction(model, processor, image: Image.Image, question: str, mode: str = 'image_only', redacted_ocr: str | None = None, max_new_tokens: int = 48) -> str:
    kind = get_model_kind(model)
    image = image.convert('RGB')
    with torch.no_grad():
        if kind == 'florence':
            prompt = f'<VQA>{question}'
            inputs = processor(text=prompt, images=image, return_tensors='pt')
            inputs = {key: value.to(device) for key, value in inputs.items()}
            generated = model.generate(**inputs, max_new_tokens=max_new_tokens)
            decoded = processor.batch_decode(generated, skip_special_tokens=True)[0]
            return sanitize_prediction_text(decoded)
        if kind == 'qwen2vl':
            prompt = build_qwen_prompt(question=question, mode=mode, redacted_ocr=redacted_ocr)
            if mode == 'image_ocr' and redacted_ocr is None:
                raise ValueError('image_ocr mode requires OCR text from generate_scenario_ocr()')
            messages = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}]
            chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[chat_text], images=[image], return_tensors='pt')
            inputs = {key: value.to(model.device) if hasattr(model, 'device') and key != 'pixel_values' else value.to(device) if hasattr(value, 'to') else value for key, value in inputs.items()}
            generated = model.generate(**inputs, max_new_tokens=max_new_tokens)
            prompt_len = inputs['input_ids'].shape[1]
            trimmed = generated[:, prompt_len:]
            decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
            return sanitize_prediction_text(decoded)
    raise ValueError(f'Unsupported model kind: {kind}')


def run_extraction(model, processor, records, scenarios, mode: str = 'image_only') -> pd.DataFrame:
    rows = []
    iterable = tqdm(records, desc=f'extraction:{get_model_kind(model)}:{mode}')
    for record in iterable:
        for scenario in scenarios:
            image = generate_scenario_image(record, scenario)
            redacted_ocr = generate_scenario_ocr(record, scenario) if mode == 'image_ocr' else None
            prediction = generate_prediction(model=model, processor=processor, image=image, question=record['question'], mode=mode, redacted_ocr=redacted_ocr)
            rows.append({
                'record_key': record['record_key'],
                'question_id': record.get('question_id', record['record_key']),
                'split': record['split'],
                'scenario': scenario,
                'prediction': prediction,
                'ocr_text_used': redacted_ocr or '',
                'ocr_source': 'generate_scenario_ocr' if mode == 'image_ocr' else '',
                'gold_answers': list(record.get('answers') or [record.get('answer', '')]),
                'coarse_type': record['coarse_field_type'],
                'question': record['question'],
            })
    return pd.DataFrame(rows)


def score_extraction_predictions(predictions_df: pd.DataFrame) -> pd.DataFrame:
    scored_rows = []
    for row in predictions_df.to_dict(orient='records'):
        record = RECORD_LOOKUP[row['record_key']]
        answers = list(row['gold_answers']) if isinstance(row['gold_answers'], list) else [str(row['gold_answers'])]
        exact_match, token_f1 = best_metric_over_answers(row['prediction'], answers)
        random_em, random_f1 = estimate_random_baseline(record, ANSWER_POOLS[row['split']])
        scored_rows.append({
            **row,
            'exact_match': exact_match,
            'token_f1': token_f1,
            'random_em': random_em,
            'random_f1': random_f1,
            'corrected_em': exact_match - random_em,
            'corrected_f1': token_f1 - random_f1,
        })
    return pd.DataFrame(scored_rows)


def compute_extraction_metrics(predictions_df: pd.DataFrame) -> pd.DataFrame:
    scored_df = score_extraction_predictions(predictions_df)
    rows = []
    for (split, scenario), group in scored_df.groupby(['split', 'scenario']):
        rows.append({
            'split': split,
            'scenario': scenario,
            'coarse_type': 'ALL',
            'n_examples': int(len(group)),
            'exact_match': float(group['exact_match'].mean()),
            'token_f1': float(group['token_f1'].mean()),
            'random_em': float(group['random_em'].mean()),
            'random_f1': float(group['random_f1'].mean()),
            'corrected_em': float(group['corrected_em'].mean()),
            'corrected_f1': float(group['corrected_f1'].mean()),
        })
    for (split, scenario, coarse_type), group in scored_df.groupby(['split', 'scenario', 'coarse_type']):
        rows.append({
            'split': split,
            'scenario': scenario,
            'coarse_type': coarse_type,
            'n_examples': int(len(group)),
            'exact_match': float(group['exact_match'].mean()),
            'token_f1': float(group['token_f1'].mean()),
            'random_em': float(group['random_em'].mean()),
            'random_f1': float(group['random_f1'].mean()),
            'corrected_em': float(group['corrected_em'].mean()),
            'corrected_f1': float(group['corrected_f1'].mean()),
        })
    return pd.DataFrame(rows).sort_values(['split', 'scenario', 'coarse_type']).reset_index(drop=True)


def sanitize_metric_name(name: str) -> str:
    return name.replace('+', '__').replace('-', '_').replace(' ', '_')


def log_csv_asset(experiment, csv_path: Path) -> None:
    experiment.log_asset(str(csv_path), file_name=csv_path.name)


def make_corrected_em_plot(metrics_df: pd.DataFrame, title: str):
    plot_df = metrics_df[metrics_df['coarse_type'] == 'ALL'].copy()
    fig, ax = plt.subplots(figsize=(10, 4))
    for split, group in plot_df.groupby('split'):
        ax.plot(group['scenario'], group['corrected_em'], marker='o', label=split)
    ax.set_title(title)
    ax.set_ylabel('Corrected EM')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig


def cleanup_model(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_model_extraction_experiment(spec: dict, scenarios: list[str], mode: str, experiment_name: str):
    experiment = comet_ml.Experiment(project_name='docvqa-privacy')
    experiment.set_name(experiment_name)
    experiment.log_parameters({'model_tag': spec['tag'], 'mode': mode, 'scenarios': scenarios, 'n_seen': len(TRAIN_RECORDS), 'n_unseen': len(VALIDATION_RECORDS), 'weights_source': 'finetuned_checkpoint', 'adapter_path': str(spec.get('final_checkpoint')) if spec['kind'] == 'qwen2vl' else '', 'ocr_source': 'generate_scenario_ocr' if mode == 'image_ocr' else 'not_used'})

    if spec['kind'] == 'florence':
        model, processor = load_florence(spec.get('final_checkpoint'))
    else:
        model, processor = load_qwen(spec['base_model_name'], spec.get('final_checkpoint'))

    predictions_df = run_extraction(model=model, processor=processor, records=ALL_RECORDS, scenarios=scenarios, mode=mode)
    scored_df = score_extraction_predictions(predictions_df)
    metrics_df = compute_extraction_metrics(predictions_df)

    prefix = f"{spec['tag']}_{mode}"
    predictions_csv = EXTRACTION_OUTPUT_DIR / f'{prefix}_predictions.csv'
    scored_csv = EXTRACTION_OUTPUT_DIR / f'{prefix}_scored.csv'
    metrics_csv = EXTRACTION_OUTPUT_DIR / f'{prefix}_metrics.csv'
    predictions_df.to_csv(predictions_csv, index=False)
    scored_df.to_csv(scored_csv, index=False)
    metrics_df.to_csv(metrics_csv, index=False)

    overall_metrics = metrics_df[metrics_df['coarse_type'] == 'ALL']
    for row in overall_metrics.to_dict(orient='records'):
        metric_name = sanitize_metric_name(f"{row['split']}_corrected_em_{row['scenario']}")
        experiment.log_metric(metric_name, row['corrected_em'])

    success_df = scored_df[(scored_df['split'] == 'seen') & (scored_df['scenario'] != 'original') & (scored_df['exact_match'] >= 1.0)].head(10).copy()
    success_csv = EXTRACTION_OUTPUT_DIR / f'{prefix}_successful_examples.csv'
    success_df.to_csv(success_csv, index=False)

    fig = make_corrected_em_plot(metrics_df, title=experiment_name)
    experiment.log_table(f'{prefix}_metrics.csv', metrics_df)
    experiment.log_table(f'{prefix}_successful_examples.csv', success_df)
    experiment.log_figure(f'{prefix}_corrected_em', fig)
    for path in [predictions_csv, scored_csv, metrics_csv, success_csv]:
        log_csv_asset(experiment, path)
    display(metrics_df.head(12))
    display(success_df)
    experiment.end()
    plt.close(fig)
    cleanup_model(model)
    return predictions_df, scored_df, metrics_df, success_df


print({'train_records': len(TRAIN_RECORDS), 'validation_records': len(VALIDATION_RECORDS), 'image_only_scenarios': IMAGE_ONLY_SCENARIOS, 'image_ocr_scenarios': QWEN_IMAGE_OCR_SCENARIOS})

## 2. Baseline Extraction Before Fine-tuning

In [ ]:
baseline_experiment = comet_ml.Experiment(project_name='docvqa-privacy')
baseline_experiment.set_name('extraction-baseline')
baseline_experiment.log_parameters({'weights_source': 'base_pretrained_model', 'adapter_path': '', 'scenario': 'img_black', 'mode': 'image_only'})
baseline_rows = []

for spec in BASELINE_SPECS:
    if spec['kind'] == 'florence':
        model, processor = load_florence()
    else:
        model, processor = load_qwen(spec['base_model_name'])
    predictions_df = run_extraction(model=model, processor=processor, records=TRAIN_RECORDS, scenarios=['img_black'], mode='image_only')
    metrics_df = compute_extraction_metrics(predictions_df)
    row = metrics_df[(metrics_df['split'] == 'seen') & (metrics_df['scenario'] == 'img_black') & (metrics_df['coarse_type'] == 'ALL')].iloc[0].to_dict()
    row['tag'] = spec['tag']
    baseline_rows.append(row)
    baseline_experiment.log_metric(f"{spec['tag']}_corrected_em_img_black", row['corrected_em'])
    cleanup_model(model)

baseline_df = pd.DataFrame(baseline_rows)
baseline_csv = EXTRACTION_OUTPUT_DIR / 'baseline_extraction_metrics.csv'
baseline_df.to_csv(baseline_csv, index=False)
baseline_experiment.log_table('baseline_extraction_metrics.csv', baseline_df)
log_csv_asset(baseline_experiment, baseline_csv)
display(baseline_df)
baseline_experiment.end()

## 3. Florence-2 Extraction (image_only)

In [ ]:
florence_spec = next(spec for spec in FINETUNED_SPECS if spec['tag'] == 'florence2')
florence_predictions_df, florence_scored_df, florence_metrics_df, florence_success_df = run_model_extraction_experiment(
    spec=florence_spec,
    scenarios=IMAGE_ONLY_SCENARIOS,
    mode='image_only',
    experiment_name='florence2-extraction',
)


## 4. Qwen2-VL-2B Extraction

In [ ]:
qwen2b_spec = next(spec for spec in FINETUNED_SPECS if spec['tag'] == 'qwen2b')
qwen2b_imageonly_outputs = run_model_extraction_experiment(
    spec=qwen2b_spec,
    scenarios=IMAGE_ONLY_SCENARIOS,
    mode='image_only',
    experiment_name='qwen2b-extraction-imageonly',
)
qwen2b_imageocr_outputs = run_model_extraction_experiment(
    spec=qwen2b_spec,
    scenarios=QWEN_IMAGE_OCR_SCENARIOS,
    mode='image_ocr',
    experiment_name='qwen2b-extraction-imageocr',
)


## 5. Qwen2-VL-7B Extraction

In [ ]:
qwen7b_spec = next(spec for spec in FINETUNED_SPECS if spec['tag'] == 'qwen7b')
qwen7b_imageonly_outputs = run_model_extraction_experiment(
    spec=qwen7b_spec,
    scenarios=IMAGE_ONLY_SCENARIOS,
    mode='image_only',
    experiment_name='qwen7b-extraction-imageonly',
)
qwen7b_imageocr_outputs = run_model_extraction_experiment(
    spec=qwen7b_spec,
    scenarios=QWEN_IMAGE_OCR_SCENARIOS,
    mode='image_ocr',
    experiment_name='qwen7b-extraction-imageocr',
)


## 6. Extraction Epoch Curve

In [ ]:
epoch_curve_rows = []
epoch_curve_experiment = comet_ml.Experiment(project_name='docvqa-privacy')
epoch_curve_experiment.set_name('extraction-epoch-curve')

for spec in FINETUNED_SPECS:
    for epoch in spec['epochs']:
        checkpoint_path = CHECKPOINTS_ROOT / spec['tag'] / f'epoch_{epoch}'
        if not checkpoint_path.exists():
            print({'tag': spec['tag'], 'epoch': epoch, 'status': 'missing_checkpoint', 'path': str(checkpoint_path)})
            continue
        if spec['kind'] == 'florence':
            model, processor = load_florence(checkpoint_path)
        else:
            model, processor = load_qwen(spec['base_model_name'], checkpoint_path)
        predictions_df = run_extraction(model=model, processor=processor, records=TRAIN_RECORDS, scenarios=['img_black'], mode='image_only')
        metrics_df = compute_extraction_metrics(predictions_df)
        row = metrics_df[(metrics_df['split'] == 'seen') & (metrics_df['scenario'] == 'img_black') & (metrics_df['coarse_type'] == 'ALL')].iloc[0]
        corrected_em = float(row['corrected_em'])
        epoch_curve_rows.append({'tag': spec['tag'], 'epoch': epoch, 'corrected_em': corrected_em})
        epoch_curve_experiment.log_metric(f"{spec['tag']}_corrected_em", corrected_em, step=epoch)
        cleanup_model(model)

epoch_curve_df = pd.DataFrame(epoch_curve_rows).sort_values(['tag', 'epoch']).reset_index(drop=True)
epoch_curve_csv = EXTRACTION_OUTPUT_DIR / 'extraction_epoch_curve.csv'
epoch_curve_df.to_csv(epoch_curve_csv, index=False)

fig, ax = plt.subplots(figsize=(8, 5))
for tag, group in epoch_curve_df.groupby('tag'):
    ax.plot(group['epoch'], group['corrected_em'], marker='o', label=tag)
ax.set_xlabel('Epoch')
ax.set_ylabel('Corrected EM')
ax.set_title('Extraction Corrected EM by Epoch (img_black, seen)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

epoch_curve_experiment.log_table('extraction_epoch_curve.csv', epoch_curve_df)
log_csv_asset(epoch_curve_experiment, epoch_curve_csv)
epoch_curve_experiment.log_figure('extraction_epoch_curve', fig)
display(epoch_curve_df)
epoch_curve_experiment.end()
plt.close(fig)

## 7. Saved Outputs

In [ ]:
saved_csvs = sorted(EXTRACTION_OUTPUT_DIR.glob('*.csv'))
saved_df = pd.DataFrame({'csv_path': [str(path) for path in saved_csvs]})
display(saved_df)
print({'output_dir': str(EXTRACTION_OUTPUT_DIR), 'csv_count': len(saved_csvs)})